# 自编码器：压缩与重建
> 难度标签：高级 | 预计时长：15-25分钟 | 前置知识：无学习经验


> 所属模块：06_深度学习 | 源文件：自编码器.py | 核心功能：编码器-解码器架构、无监督特征学习

## 概述

自编码器通过"压缩-解压"学习数据的低维表示。编码器将输入压缩到瓶颈层，解码器从瓶颈层重建输入。训练目标是最小化重建误差。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
# --- 导入库 ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 数学原理

### 1. 自编码器的基本结构

自编码器由**编码器** $f_\theta$ 和**解码器** $g_\phi$ 组成，学习一个压缩-重建映射：

$$z = f_\theta(x), \quad \hat{x} = g_\phi(z)$$

目标是最小化重建误差：

$$\mathcal{L} = \|x - \hat{x}\|^2 = \|x - g_\phi(f_\theta(x))\|^2$$

### 2. 线性自编码器与 PCA 的等价性

当编码器和解码器都是线性变换时（单隐层、无激活函数）：

$$z = W_e x + b_e, \quad \hat{x} = W_d z + b_d$$

最小化 MSE 损失等价于 PCA。$W_d$ 的行空间张成数据协方差矩阵 $\Sigma = \frac{1}{n}X^TX$ 的前 $k$ 个主成分方向。

### 3. 欠完备与过完备

**欠完备**（undercomplete）：$\dim(z) < \dim(x)$，强制学习数据的主要结构：

$$z \in \mathbb{R}^d, \quad d < D = \dim(x)$$

代码中 `latent_dim=16 < input_dim=64`，压缩比 $4:1$。

**过完备**（overcomplete）：$\dim(z) \geq \dim(x)$，需要额外正则化（稀疏性、噪声）防止恒等映射。

### 4. MSE 重建损失

代码中 `nn.MSELoss()` 计算：

$$\mathcal{L}_{MSE} = \frac{1}{D}\sum_{i=1}^{D}(x_i - \hat{x}_i)^2$$

假设重建误差服从各向同性高斯分布 $p(x|\hat{x}) = \mathcal{N}(\hat{x}, \sigma^2 I)$，最小化 MSE 等价于最大似然估计。

### 5. 编码器的逐层压缩

代码中编码器结构为 $64 \to 128 \to 64 \to 16$，数学上是复合映射：

$$z = W_3 \sigma(W_2 \sigma(W_1 x + b_1) + b_2) + b_3$$

其中 $\sigma = \text{ReLU}$。每层的权重矩阵维度：

| 层 | 输入维度 | 输出维度 | 功能 |
|----|----------|----------|------|
| Layer 1 | 64 | 128 | 扩展到高维特征空间 |
| Layer 2 | 128 | 64 | 压缩回原维度 |
| Layer 3 | 64 | 16 | 最终压缩到潜在空间 |

### 6. 解码器的对称重建

解码器结构 $16 \to 64 \to 128 \to 64$，对称地逐步恢复维度：

$$\hat{x} = \sigma_{out}(W_6 \sigma(W_5 \sigma(W_4 z + b_4) + b_5) + b_6)$$

输出层用 $\sigma_{out} = \text{Sigmoid}$ 将值映射到 $[0, 1]$（配合标准化后的输入数据）。

### 7. 潜在空间的几何性质

训练好的自编码器，潜在空间 $z \in \mathbb{R}^{16}$ 具有以下性质：
- 相似输入的编码在潜在空间中聚集（局部连续性）
- 潜在维度的每一维可能对应某种语义特征
- 但**不保证**潜在空间的全局规则性（这是 VAE 的改进目标）

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `encoder`: Linear(64→128→64→16) | $z = f_\theta(x)$ 编码映射 |
| `decoder`: Linear(16→64→128→64) | $\hat{x} = g_\phi(z)$ 解码映射 |
| `ReLU()` | $\sigma(x) = \max(0, x)$ 非线性激活 |
| `Sigmoid()` 输出 | $\sigma(x) = \frac{1}{1+e^{-x}} \in (0,1)$ |
| `MSELoss` | $\frac{1}{D}\sum(x_i - \hat{x}_i)^2$ |
| `latent_dim=16` | 潜在空间维度，控制压缩程度 |
| `nn.MSELoss()` 用于重建质量评估 | 测试集上的泛化重建误差 |

### 1. 加载数据

首先加载数据集，为后续训练和评估做准备。

In [ ]:
digits = load_digits()
X = StandardScaler().fit_transform(digits.data)
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)
X_train_t = torch.FloatTensor(X_train)
X_test_t = torch.FloatTensor(X_test)
# --- 继续 ---
train_loader = DataLoader(TensorDataset(X_train_t, X_train_t), batch_size=64, shuffle=True)

### 2. 自编码器模型

运行 2. 自编码器模型 的代码，观察算法在该环节的行为。

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim=64, latent_dim=16):
        super().__init__()
        # 编码器
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
# --- 继续 ---
            nn.Linear(64, latent_dim),
        )
        # 解码器
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
# --- 继续 ---
            nn.Linear(128, input_dim),
            nn.Sigmoid(),  # 输出范围 [0,1]
        )

    def forward(self, x):
        z = self.encoder(x)
        x_recon = self.decoder(z)
        return x_recon, z

model = Autoencoder(input_dim=64, latent_dim=16)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print(f"=== 自编码器模型 ===")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"总参数量: {total_params:,}")

### 3. 训练

执行模型训练循环，观察损失函数的收敛过程。

In [ ]:
print("\n=== 训练 ===")
n_epochs = 50
for epoch in range(n_epochs):
    model.train()
    total_loss = 0
# --- 循环处理 ---
    for batch_X, _ in train_loader:
        recon, _ = model(batch_X)
        loss = criterion(recon, batch_X)
        optimizer.zero_grad()
        loss.backward()
# --- 继续 ---
        optimizer.step()
        total_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_recon, _ = model(X_test_t)
            test_loss = criterion(test_recon, X_test_t).item()
# --- 输出结果 ---
        print(f"  Epoch {epoch+1:>2}: Train Loss={total_loss/len(train_loader):.6f}, "
              f"Test Loss={test_loss:.6f}")

### 4. 重建质量

运行 4. 重建质量 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 重建效果 ===")
model.eval()
with torch.no_grad():
    test_recon, _ = model(X_test_t)
    # 逐样本重建误差
    errors = ((X_test_t - test_recon) ** 2).mean(dim=1)
    print(f"平均重建误差: {errors.mean():.6f}")
    print(f"最大重建误差: {errors.max():.6f}")
    print(f"最小重建误差: {errors.min():.6f}")

### 5. 不同潜在维度

运行 5. 不同潜在维度 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 不同 latent_dim 对比 ===")
for ld in [2, 4, 8, 16, 32]:
    m = Autoencoder(input_dim=64, latent_dim=ld)
    opt = optim.Adam(m.parameters(), lr=1e-3)
    for _ in range(50):
# --- 继续 ---
        m.train()
        for bx, _ in train_loader:
            recon, _ = m(bx)
            loss = criterion(recon, bx)
            opt.zero_grad()
# --- 继续 ---
            loss.backward()
            opt.step()
    m.eval()
    with torch.no_grad():
        test_recon, _ = m(X_test_t)
# --- 继续 ---
        test_loss = criterion(test_recon, X_test_t).item()
    n_params = sum(p.numel() for p in m.parameters())
    print(f"  latent_dim={ld:>2}: Test MSE={test_loss:.6f}, 参数量={n_params:,}")

### 6. 去噪自编码器

运行 6. 去噪自编码器 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== 去噪自编码器 (Denoising Autoencoder) ===")
noise_factor = 0.5
X_train_noisy = X_train_t + noise_factor * torch.randn_like(X_train_t)
X_test_noisy = X_test_t + noise_factor * torch.randn_like(X_test_t)

denoise_model = Autoencoder(input_dim=64, latent_dim=16)
opt_d = optim.Adam(denoise_model.parameters(), lr=1e-3)
train_loader_d = DataLoader(TensorDataset(X_train_noisy, X_train_t), batch_size=64, shuffle=True)

for epoch in range(50):
    denoise_model.train()
    for noisy, clean in train_loader_d:
        recon, _ = denoise_model(noisy)
        loss = criterion(recon, clean)
# --- 继续 ---
        opt_d.zero_grad()
        loss.backward()
        opt_d.step()

denoise_model.eval()
with torch.no_grad():
    noisy_recon, _ = denoise_model(X_test_noisy)
    denoise_loss = criterion(noisy_recon, X_test_t).item()
    # 对比直接用原始自编码器处理噪声数据
    clean_recon, _ = model(X_test_noisy)
    clean_loss = criterion(clean_recon, X_test_t).item()
print(f"去噪自编码器重建损失: {denoise_loss:.6f}")
print(f"普通自编码器处理噪声: {clean_loss:.6f}")

print("\n=== 自编码器要点 ===")
print("- 瓶颈层（latent_dim）迫使网络学习数据的压缩表示")
print("- 去噪自编码器：输入加噪声，目标是原始数据，增强鲁棒性")
print("- 可用于：降维、去噪、特征学习、数据生成（变体）")
print("- latent_dim 太小 → 信息丢失；太大 → 过拟合/退化为恒等映射")

## 关键代码解释

```python
# 编码器: 输入 -> 瓶颈
encoder = nn.Sequential(nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 32))
# 解码器: 瓶颈 -> 重建
decoder = nn.Sequential(nn.Linear(32, 256), nn.ReLU(), nn.Linear(256, 784))
```

## 注意事项

1. **线性自编码器 = PCA**
2. **瓶颈层维度选择**：太小信息丢失，太大无压缩效果
3. **可用于去噪**：输入加噪声，目标是干净数据

## 总结与延伸

以上代码展示了 **自编码器** 的完整流程。

**解读要点**：
- 关注 **损失函数** 的收敛曲线
- 观察训练/验证损失是否出现分叉（过拟合信号）
- 对比不同超参数（学习率、层数）的影响

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **变分自编码器（VAE）**：概率版本，可生成新样本
- **稀疏自编码器**：加入稀疏约束
- **自编码器用于异常检测**：重建误差大的样本可能是异常
